# Baseline search: BM25 over page text
#### https://www.youtube.com/watch?v=TW9vHU1GpU4 quick watch on BM25 

#### Short Notes:
- Q1: How many times do these words appear in each document?
- Q2: Are they rare or common across the whole collection ?
- Q3: Are we looking at a short, focussed document or a huge blob of text? 

### BM25 score of page $D$ for question $Q$

![BM25 formula, annotated](BM25-figure.jpg)

| metric | question it answers |
|---|---|
| Hit@5 | is at least one correct page in the top 5? |
| Recall@5 | what share of all correct pages is in the top 5? |
| MRR@10 | how high is the first correct page? (1 = rank 1, 0.5 = rank 2, ...) |
| nDCG@10 | are the best pages (score 2) near the top? 1.0 = perfect order |

## 1. Load dev questions and page texts

In [23]:
import json
import math
from collections import Counter
from pathlib import Path

import bm25s
from datasets import load_dataset

EVALS = Path("../evals") if Path("../evals").is_dir() else Path("evals")

# dev.jsonl = one JSON object per line -> json.loads on each line
dev = [json.loads(line) for line in (EVALS / "dev.jsonl").read_text().splitlines()]

corpus = load_dataset("vidore/vidore_v3_pharmaceuticals", "corpus")["test"]
corpus = corpus.select_columns(["corpus_id", "markdown"])      # no images: search reads text only
page_ids = corpus["corpus_id"]                                 # position in the list -> corpus_id
page_texts = corpus["markdown"]

print("dev questions:", len(dev))
print("pages:", len(page_ids), "| empty pages:", sum(not t.strip() for t in page_texts))

dev questions: 108
pages: 2313 | empty pages: 0


In [9]:
q = dev[0]
print("question:", q["query"])
print("answer:  ", q["answer"])
print("correct pages (corpus_id: score):", {p["corpus_id"]: p["score"] for p in q["pages"]})

question: Does the Bolar exception help reduce the time it takes for generic drugs to reach the market after a patent expires?
answer:   Yes, the Bolar exception helps reduce the time it takes for generic drugs to reach the market after a patent expires by allowing generic manufacturers to conduct the necessary studies and trials for regulatory approval while the original drug's patent is still active.
correct pages (corpus_id: score): {331: 2, 332: 1, 333: 2, 335: 2, 336: 2, 337: 2, 338: 1, 339: 2, 341: 1, 342: 1, 343: 1, 344: 2, 351: 2, 521: 1, 549: 1, 560: 1}


## 2. Build the index

`tokenize` lowercases, splits into words and drops stop words ("the", "of", ...). `index` counts every word on every page.

In [18]:
retriever = bm25s.BM25()
retriever.index(bm25s.tokenize(page_texts, stopwords="english"))

## 3. Search one question

`retrieve` returns **positions** in `page_texts` (not `corpus_id`s) and their scores, one row per question.

In [19]:
positions, scores = retriever.retrieve(bm25s.tokenize([q["query"]], stopwords="english"), k=5)

correct = {p["corpus_id"]: p["score"] for p in q["pages"]}
print(q["query"], "\n")
for rank, (pos, score) in enumerate(zip(positions[0], scores[0]), start=1):
    cid = page_ids[pos]
    mark = f"CORRECT (score {correct[cid]})" if cid in correct else ""
    print(f"{rank}. corpus_id {cid:5}  bm25 {score:6.2f}  {mark}")
    print("   ", page_texts[pos][:150].replace("\n", " "), "\n")

List down the objectives of the New Drugs Regulatory Program Modernization, and the Clinical Offices and Division Names. 

1. corpus_id  1776  bm25   9.98  
    The New Drugs Regulatory Program has 6 active initiatives FDA Integrated Review for Marketing Applications Developing a streamlined interdisciplinary  

2. corpus_id  1777  bm25   9.54  
    Re-org will therapeutically align offices and divisions to support scientific exchange and improve consistency across OND Approved OND Director eeputy 

3. corpus_id  1778  bm25   8.20  CORRECT (score 1)
    FDA Clinical Office and Division Names Office of Regulatory Operation: Divisions of Regulatory Operations Approved Office of Rare Diseases, Pediatrics 

4. corpus_id  1775  bm25   7.99  CORRECT (score 1)
    New Drugs Regulatory Program Modernization FDA Objectives Guiding principles for modernizing the new drugs regulatory program We will grow our scienti 

5. corpus_id  1769  bm25   7.91  
    FDA Topics to be covered New drug review 

## 4. Metrics

Each function takes `ranked` (corpus_ids, best first) and `grades` (`{corpus_id: score}` of the correct pages).

In [20]:
def hit_at_k(ranked, grades, k):
    return float(any(c in grades for c in ranked[:k]))

def recall_at_k(ranked, grades, k):
    return len(set(ranked[:k]) & set(grades)) / len(grades)

def mrr_at_k(ranked, grades, k):
    for rank, c in enumerate(ranked[:k], start=1):
        if c in grades:
            return 1 / rank
    return 0.0

def ndcg_at_k(ranked, grades, k):
    # gain = the page's score (2 or 1), discounted by log2 of its position; divided by the best possible order
    dcg = sum(grades.get(c, 0) / math.log2(i + 2) for i, c in enumerate(ranked[:k]))
    ideal = sum(g / math.log2(i + 2) for i, g in enumerate(sorted(grades.values(), reverse=True)[:k]))
    return dcg / ideal

## 5. Search every dev question and average

In [21]:
positions, scores = retriever.retrieve(bm25s.tokenize([q["query"] for q in dev], stopwords="english"), k=10)

results = []                                                   # one row per question, kept for failure analysis
for q, row in zip(dev, positions):
    ranked = [page_ids[p] for p in row]
    grades = {p["corpus_id"]: p["score"] for p in q["pages"]}
    results.append({
        "query_id": q["query_id"], "query": q["query"], "ranked": ranked, "grades": grades,
        "content_type": q["content_type"], "query_types": q["query_types"],
        "hit@5": hit_at_k(ranked, grades, 5), "recall@5": recall_at_k(ranked, grades, 5),
        "mrr@10": mrr_at_k(ranked, grades, 10), "ndcg@10": ndcg_at_k(ranked, grades, 10),
    })

metrics = {m: sum(r[m] for r in results) / len(results) for m in ["hit@5", "recall@5", "mrr@10", "ndcg@10"]}
for m, v in metrics.items():
    print(f"{m:9} {v:.3f}")

hit@5     0.833
recall@5  0.509
mrr@10    0.655
ndcg@10   0.564


## 6. Where does it miss?

Questions with **no** correct page in the top 5. Look for a pattern: charts and tables? Numbers? Words that differ between
question and page?

In [22]:
misses = [r for r in results if r["hit@5"] == 0]
print(f"missed {len(misses)} of {len(results)} questions\n")

print("content type of the misses:", Counter(t for r in misses for t in r["content_type"]))
print("content type of all dev:   ", Counter(t for r in results for t in r["content_type"]), "\n")

for r in misses[:5]:
    print("Q:", r["query"])
    print("   correct:", r["grades"], "| top 5:", r["ranked"][:5], "\n")

missed 18 of 108 questions

content type of the misses: Counter({'Text': 17, 'Chart': 7, 'Infographic': 5, 'Table': 4, 'Other': 1, 'Image': 1})
content type of all dev:    Counter({'Text': 101, 'Infographic': 31, 'Chart': 23, 'Table': 19, 'Image': 10, 'Other': 1}) 

Q: Analyze trends in NDA and BLA approval rates from 2010 to 2019 and their implications for regulatory efficiency
   correct: {1761: 1, 1762: 1, 1763: 1, 1764: 1, 1789: 1} | top 5: [1786, 154, 1234, 1835, 1853] 

Q: Examine the patterns in global first-in-the-world drug approvals and explore the reasons behind the United States' leading role relative to the European Union and Japan.
   correct: {2149: 1, 2150: 1, 2152: 1, 2153: 1, 2154: 1, 2167: 1, 2168: 1, 2169: 1, 2170: 1} | top 5: [1535, 630, 386, 1340, 1648] 

Q: How do patent linkage and data exclusivity differ in their impact on generic drug approval and access to affordable medicines?
   correct: {385: 1, 525: 1, 533: 1, 546: 1, 550: 1} | top 5: [556, 547, 549, 565,

## 7. Log the run

Every run writes its config and numbers to `reports/runs/` (see `reports/README.md`).

In [15]:
from vdoc_agent.runlog import log_run

config = {"retriever": "bm25s", "text": "markdown", "stopwords": "english", "split": "dev", "k": 10}
print(log_run("bm25-v0", config, metrics))

/storage/homefs/kr23w045/vdoc-agent/reports/runs/20261007T001533Z_bm25-v0.json
